In [ ]:
%cd ../..

In [ ]:
import sys
from pathlib import Path

import polars as pl
from loguru import logger

In [ ]:
logger.remove()
logger.add(sys.stderr, level='INFO')

# Read data

In [ ]:
paths = Path("data/raw/pos/").glob("**/*.csv")

cols_name = [
    'date',
    'time',
    'restaurant',
    'meal_type',
    'name',
    'pcs',
    'co2',
]

df_list = []
for path in paths:
    df = pl.read_csv(path, separator=';')
    df.columns = cols_name
    df = df.with_columns(pl.lit(path.stem).alias('src'))

    df_list.append(df)

pos_raw = pl.concat(df_list)
pos_raw.head()

In [ ]:
path = "data/processed/dim_restaurants.xlsx"
dim_restaurants = pl.read_excel(path)

dim_restaurants.head()

In [ ]:
path = "data/processed/dim_meal_types.xlsx"
dim_meal_types = pl.read_excel(path)

dim_meal_types.head()

In [ ]:
path = "data/processed/dim_meals.parquet"
dim_meals = (
    pl.read_parquet(path)
    .select(
        pl.col('id').alias('meal_id'),
        'meal_codes',
        'names'
    )
    .explode('meal_codes')
    .explode('names')
    .rename({
        'meal_codes': 'meal_code',
        'names': 'name'
    })
)
dim_meals.head()

# Process

In [ ]:
pat_meal_code = r"(\d{1,})"
pat_meal_code_replaced = r"(?:\d{1,})\s"

cols = ['id', 'restaurant', 'meal_id', 'datetime', 'pcs', 'src']

pos = (
    pos_raw

    # Transform from existing columns
    .with_columns(
        pl.concat_str('date', 'time', separator="|").str.to_datetime("%d.%m.%Y|%H:%M").alias('datetime'),
        pl.col('name').str.extract(pat_meal_code).alias('meal_code').cast(pl.Int32()),
        pl.col('name').str.strip_chars(),
    )

    # Add other info from dim tables
    .join(dim_meal_types, on='meal_type', how='left')
    .join(dim_restaurants, on='restaurant', how='left')
    .join(dim_meals, on='name', how='left')
    .join(dim_meals, on='meal_code', how='left')
    


    # Post-process
    .select(
        pl.col('restaurant_id').alias('restaurant'),
        pl.coalesce('meal_id', 'meal_id_right').alias('meal_id'),
        'datetime',
        pl.col('pcs').fill_null(0),
        'src'
    )
    .with_row_index('id')

    .select(cols)
)

pos.head()

## Sanity check

In [ ]:
assert pos.filter(pl.col('meal_id').is_null()).shape[0] == 0
assert True not in pos.select(pl.all().has_nulls()).rows()[0]

# Save

In [ ]:
path = "data/processed/pos.xlsx"

pos.write_excel(path)